# Week 5b Hands-On Lab — Prompting and Evaluating a Video World Model

**ESP3201 · formative hands-on lab.** Generation needs a free-tier Colab **GPU (T4) runtime**. If no GPU is available, use the **precomputed rollout-bank** fallback — you still do the full evaluation on real frames.

A video-generation model is a **world model**: a learned simulator of how a scene evolves. You will vary **prompting**, **input controls** (seed, frames, guidance), and **evaluation criteria**, and measure how each affects the quality, consistency, and *usefulness* of the rollout.

> **Report only frames your run actually produced** (or the provided rollout-bank clips). No projected results.

**Attribution.** Evaluation dimensions follow **VBench** (Huang et al., 2023); generation follows the **diffusers** text-to-video tutorials. Lab code is original to ESP3201.

## Setup

In [ ]:
import sys, os
from IPython.display import display

COURSE_REPO_URL = "https://github.com/bingquan/esp3201-public.git"

os.system('pip install -q numpy pillow')
try:
    import video_eval
except ModuleNotFoundError:
    cand = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
    if os.path.exists(os.path.join(cand, 'video_eval.py')):
        sys.path.insert(0, cand)
    elif COURSE_REPO_URL:
        os.system(f'git clone -q {COURSE_REPO_URL} course_repo')
        sys.path.insert(0, 'course_repo/labs/week05_embodied_system_critique/starter')
    else:
        raise ModuleNotFoundError('video_eval.py not found. On Colab set COURSE_REPO_URL.')
    import video_eval
from video_eval import (score_clip, synth_clip, load_frames_from_dir,
                        frames_from_diffusers, show_frames, compare_frames,
                        mean_abs_diff)
print('video_eval loaded.')

## Warm-up — what the metrics mean (no model needed)

Three synthetic clips show how the objective metrics behave: a static clip, a smoothly moving object, and pure noise.

In [ ]:
for kind in ('static', 'moving', 'noise'):
    frames = synth_clip(kind)
    print(score_clip(frames, label=kind))
    display(show_frames(frames, n=8))

**What `video_eval` is actually computing.** Every metric above is a plain, offline comparison between adjacent frames — no model, no learned scorer, nothing that understands what an "arm" or a "cube" is:

- `temporal_consistency` = 1 − mean absolute pixel-value change between consecutive frames. High = visually stable.
- `motion_magnitude` is that same mean absolute change — the complement of consistency (`consistency = 1 − motion`).
- `flicker` is the standard deviation, over time, of each frame's overall brightness — a proxy for *global* instability (e.g. the whole frame suddenly darkening or flashing), not local object motion.
- `brightness_drift` is how far a frame's brightness ever wanders from frame 0, at its worst point.

Look at the strip above alongside the numbers: **static** is consistency ≈ 1 / motion ≈ 0 (frozen); **moving** has real but small motion with near-zero flicker (smooth, coherent); **noise** has high motion *and* high flicker (incoherent — nothing to track frame to frame). That is the whole diagnostic: high consistency with near-zero motion means *nothing is happening*; high flicker means *global incoherence*. A useful world-model rollout needs motion that is consistent, not frozen and not chaotic — and because these are pixel statistics only, they cannot tell you whether the motion is *physically correct*, which is why the worksheet also asks for your own human judgment (prompt adherence, identity stability, physical plausibility).

## Generate clips with a real video model (Colab GPU)

Pinned to **`cerspense/zeroscope_v2_576w`**, **verified on a free T4** (fp16 + CPU offload, measured ~5.8 GB peak; see `pinned_models.md`). Swap `MODEL_ID` and re-check memory if you like.

> **Time budget (important).** On a free T4 each clip takes **roughly 1–3 minutes** (much slower than the lab's reference GPU). The default below runs **3 experiments**, which is enough to see the controls matter; only add more if your session has time. Generate one clip first to gauge your runtime before launching the grid.

In [ ]:
# Run on a GPU runtime.
os.system('pip install -q diffusers transformers accelerate torch')
import torch
from diffusers import DiffusionPipeline

MODEL_ID = 'cerspense/zeroscope_v2_576w'   # VERIFIED on free T4 (~5.8 GB fp16+offload)
pipe = DiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
try:
    pipe.enable_vae_slicing()
except Exception:
    pass

def generate(prompt, seed=0, num_frames=24, num_inference_steps=25,
             guidance_scale=9.0, negative_prompt=None):
    g = torch.Generator(device='cuda').manual_seed(seed)
    out = pipe(prompt, num_frames=num_frames, num_inference_steps=num_inference_steps,
               guidance_scale=guidance_scale, negative_prompt=negative_prompt,
               height=320, width=576, generator=g)
    return frames_from_diffusers(out)   # zeroscope returns numpy frames; adapter handles it

print('peak VRAM after load (GB):', round(torch.cuda.max_memory_allocated()/1e9, 2))

### Controlled experiments (3 by default)

Each changes **one factor** so you can attribute the effect:

1. **terse** vs **detailed** prompt — does detail + motion verbs help?
2. **detailed + negative prompt** — does suppressing 'blurry, distorted, flickering' clean it up?

Once these run, the worksheet invites you to add a **seed** change and a **guidance** change *if time allows*.

In [ ]:
experiments = {
  'terse':         dict(prompt='a robot arm picking up a cube'),
  'detailed':      dict(prompt='a robotic arm slowly grasping a red cube on a table, smooth motion, realistic'),
  'detailed+neg':  dict(prompt='a robotic arm slowly grasping a red cube on a table, smooth motion, realistic',
                        negative_prompt='blurry, distorted, flickering'),
}
scores = []
for label, kw in experiments.items():
    frames = generate(**kw)
    scores.append(score_clip(frames, label=label))
    print(scores[-1])
    # A labeled frame strip you can screenshot/right-click-save straight into
    # your report worksheet — cite the frame numbers it shows when you write
    # up what changed.
    display(show_frames(frames, n=8))

### Optional extra controls (only if your session has time)

```python
# same prompt, different seed -> how stable is the model?
print(score_clip(generate('a robot arm picking up a cube', seed=1), label='seedB'))
# higher guidance -> tighter prompt adherence, but can over-sharpen / flicker
print(score_clip(generate('a robot arm picking up a cube', guidance_scale=14.0), label='high_guidance'))
```

## No GPU? Use the precomputed rollout bank

Evaluate real generated frames committed under `rollout_bank/` without generating them yourself.

In [ ]:
import glob
base = next((p for p in ['../rollout_bank',
            'course_repo/labs/week05_embodied_system_critique/rollout_bank']
            if glob.glob(p + '/clip*')), None)
for d in sorted(glob.glob((base or '../rollout_bank') + '/clip*')):
    frames = load_frames_from_dir(d)
    print(score_clip(frames, label=d.split('/')[-1]))
    display(show_frames(frames, n=8))

## Part 2 — Opening the Box: How the Model Actually Works

Part 1 treated the model as a black box and scored what came out. Part 2 pries the box open: you'll look at the VAE that compresses each frame, and at what the diffusion sampler's own controls (step count, guidance) and its starting noise actually do.

**Honest architecture note.** The lecture's "causal 3D VAE" describes newer-generation video models. `zeroscope_v2_576w` is an older-generation ModelScope-style pipeline: its VAE is the same **per-frame 2D image VAE** used in Stable Diffusion (no temporal awareness at all) — all temporal structure lives in the U-Net's 3D convolutions, not the VAE. That contrast is itself worth reporting: it means this pinned model's VAE round-trip result tells you about *per-frame* compression only, not about how it handles *time*.

**Compute/track note.** Exercise 1 (VAE round-trip) runs on CPU with no full pipeline needed, so it works for both tracks. Exercises 2–4 need the full GPU pipeline from Part 1's generation cell (Track A only) — Track B students note this in the worksheet instead of running them.

### Exercise 1 — VAE round-trip

Encode one real frame through the VAE, decode it straight back, and compare. This is the cheapest possible experiment here (a couple of forward passes, no iterative denoising) and it makes "the VAE is a lossy compressor" into something you can see: blur, lost fine texture, color drift.

In [ ]:
import torch
import numpy as np
import glob

# Works whether or not you ran the GPU generation cells above.
try:
    vae = pipe.vae.to('cuda' if torch.cuda.is_available() else 'cpu')
    # ^ pipe.vae's own .device attribute is unreliable under enable_model_cpu_offload():
    # it can claim 'cuda' while the weights are actually still sitting on CPU (the
    # offload hook only fires on the pipeline's forward(), not on a raw .encode()/.decode()
    # call). Forcing it explicitly here avoids a "tensors on different devices" error.
    print('Using the VAE from the already-loaded pipeline.')
except NameError:
    from diffusers import AutoencoderKL
    vae = AutoencoderKL.from_pretrained('cerspense/zeroscope_v2_576w', subfolder='vae')
    print('No pipeline loaded -- loaded just the VAE component (CPU, no GPU needed for this exercise).')

def vae_roundtrip(frame_uint8):
    """Encode one real frame through the VAE and decode it straight back."""
    device = next(vae.parameters()).device
    x = torch.from_numpy(frame_uint8.astype(np.float32) / 127.5 - 1.0)  # -> [-1, 1]
    x = x.permute(2, 0, 1).unsqueeze(0).to(vae.dtype).to(device)
    with torch.no_grad():
        latent = vae.encode(x).latent_dist.sample() * vae.config.scaling_factor
        recon = vae.decode(latent / vae.config.scaling_factor).sample
    recon = ((recon.clamp(-1, 1) + 1) * 127.5).squeeze(0).permute(1, 2, 0).cpu().numpy().astype(np.uint8)
    return recon, latent

# One real frame: your own generation if you ran one above, else a rollout-bank frame.
try:
    src_frame = frames[0]
except NameError:
    bank_clip = sorted(glob.glob('../rollout_bank/clip*'))[0]
    src_frame = load_frames_from_dir(bank_clip)[0]

recon_frame, latent = vae_roundtrip(src_frame)
compression_ratio = (src_frame.shape[0] * src_frame.shape[1] * src_frame.shape[2]) / latent.numel()
print(f'original frame: {src_frame.shape}  ->  latent: {tuple(latent.shape)}')
print(f'compression ratio (element count): {compression_ratio:.1f}x')
print('pixel-space reconstruction error (mean abs diff, 0-1 scale):',
      round(mean_abs_diff([src_frame, recon_frame]), 4))
display(compare_frames({'original': src_frame, 'VAE reconstruction': recon_frame}))

### Exercise 2 — Diffusion step-count sweep (GPU / Track A)

The pinned setting is 25 inference steps. Fewer steps means less iterative denoising — same starting noise, same prompt, just less time spent refining it. This costs one more generation per value (~1-3 min each on a free T4).

In [ ]:
# GPU required (Track A) -- needs `generate` from the Part 1 pipeline cell.
step_sweep = {}
for steps in (5, 15, 25):
    frames_i = generate('a robot arm picking up a cube', num_inference_steps=steps)
    step_sweep[steps] = score_clip(frames_i, label=f'{steps} steps')
    print(step_sweep[steps])
    display(show_frames(frames_i, n=8))

### Exercise 3 — Guidance-scale sweep (GPU / Track A)

Guidance scale controls how hard the model is pushed to match the text prompt versus its own unconditional prior. Low guidance drifts from the prompt; very high guidance can over-sharpen or flicker (the pinned default is 9.0).

In [ ]:
# GPU required (Track A) -- needs `generate` from the Part 1 pipeline cell.
guidance_sweep = {}
for g in (1.0, 9.0, 15.0):
    frames_i = generate('a robot arm picking up a cube', guidance_scale=g)
    guidance_sweep[g] = score_clip(frames_i, label=f'guidance={g}')
    print(guidance_sweep[g])
    display(show_frames(frames_i, n=8))

### Exercise 4 — Latent-space interpolation (GPU / Track A)

Every generation starts from a random noise tensor (the "latent" the diffusion sampler denoises step by step). Here we build two different starting-noise tensors from two seeds and walk between them with **slerp** (spherical interpolation), running the **full** sampler from each interpolated starting point with the same prompt.

**Why slerp, not a plain average.** Two independent random Gaussian noise tensors are very close to orthogonal in this many dimensions, so their plain average (lerp) has a *smaller norm* than either one — verified here: norm(A) ≈ 526, norm(B) ≈ 526, but norm(lerp(A,B,0.5)) ≈ 372. Decoding that shrunk, off-manifold point produces a flat, nearly featureless frame — not a meaningful "in-between" scene, just a degenerate one. Slerp keeps the norm constant along the path (≈526 throughout), which is why it's the standard recipe for latent-space walks: it stays on the sphere the model was actually trained to denoise from.

In [ ]:
# GPU required (Track A). Verified against the pinned checkpoint. Learns the
# exact latent tensor shape (rather than assuming it) via a cheap 1-step probe.
try:
    PROMPT = 'a robot arm picking up a cube'
    _shape_probe = {}

    def _grab_shape(step, timestep, latents):
        # NOTE: this pipeline (TextToVideoSDPipeline) is on the OLDER diffusers
        # callback API -- callback(step, timestep, latents), not the newer
        # callback_on_step_end(pipe, step, timestep, callback_kwargs). If you
        # swap MODEL_ID for a newer pipeline, check its __call__ signature
        # (inspect.signature(pipe.__call__)) before assuming either style.
        _shape_probe['shape'] = latents.shape
        _shape_probe['dtype'] = latents.dtype
        _shape_probe['device'] = latents.device

    # Must match height/width used below, or this silently probes the
    # pipeline's *default* resolution instead of the pinned 320x576.
    _ = pipe(PROMPT, num_frames=24, num_inference_steps=1, height=320, width=576,
             callback=_grab_shape, callback_steps=1)
    shape, dtype, device = _shape_probe['shape'], _shape_probe['dtype'], _shape_probe['device']
    print('learned latent shape:', tuple(shape))

    def make_init_latents(seed):
        g = torch.Generator(device=device).manual_seed(seed)
        return torch.randn(shape, generator=g, device=device, dtype=dtype)

    def slerp(a, b, t):
        """Spherical interpolation -- keeps the norm constant (see markdown above)."""
        a_f, b_f = a.flatten().float(), b.flatten().float()
        a_n, b_n = a_f / a_f.norm(), b_f / b_f.norm()
        omega = torch.acos((a_n * b_n).sum().clamp(-1, 1))
        if omega.abs() < 1e-4:
            return (1 - t) * a + t * b
        so = torch.sin(omega)
        out = (torch.sin((1 - t) * omega) / so) * a_f + (torch.sin(t * omega) / so) * b_f
        return out.reshape(a.shape).to(a.dtype)

    latent_A = make_init_latents(0)
    latent_B = make_init_latents(7)
    print('norm A:', round(latent_A.float().norm().item(), 1),
          ' norm B:', round(latent_B.float().norm().item(), 1))

    interp_frames = {}
    for t in (0.0, 0.5, 1.0):
        mixed = slerp(latent_A, latent_B, t)
        out = pipe(PROMPT, num_frames=24, num_inference_steps=25,
                   guidance_scale=9.0, height=320, width=576, latents=mixed)
        fr = frames_from_diffusers(out)
        interp_frames[f't={t}'] = fr[0]   # first frame of each clip, side by side
        print(f't={t}:', score_clip(fr, label=f't={t}'))
    display(compare_frames(interp_frames))
except Exception as e:
    print('This cell needs the live GPU pipeline from Part 1. If your diffusers version '
          'differs from the one this was verified against, the callback API may have '
          'changed again -- check inspect.signature(pipe.__call__) before assuming.')
    print('Error (report this verbatim in your write-up, do not guess what it would show):')
    print(repr(e))

## Worksheet (your deliverable)

### 1. Controls x criteria grid

Record objective metrics + your 1-5 human judgments for each experiment you ran:

| Experiment | Temporal consistency | Motion | Flicker | Prompt adherence (1-5) | Identity stability (1-5) | Physical plausibility (1-5) |
|------------|----------------------|--------|---------|------------------------|--------------------------|-----------------------------|
| terse | | | | | | |
| detailed | | | | | | |
| detailed+neg | | | | | | |
| *(optional)* seedB | | | | | | |
| *(optional)* high_guidance | | | | | | |

### 2. Which controls mattered?

- Which single control most improved a criterion you care about, with the numbers/judgments that show it?
- Where did the metrics and your eyes **disagree** (e.g. high consistency but the object morphed)? What does that say about trusting any single metric?

### 3. World-model-as-predictor

- At what `num_frames` / horizon did coherence break for your model?
- `Would you trust this model's rollout inside a planning loop? For what horizon, and why?`

### 4. Part 2 — Opening the Box (required)

**4a. VAE round-trip.** Report: original frame shape, latent shape, compression ratio, mean-abs-diff reconstruction error, and embed the original/reconstruction comparison image. In 1-2 sentences: what specifically got lost (fine texture? color? edges?) and does that match "the VAE is a per-frame 2D compressor with no notion of time" from the intro note?

**4b. Diffusion step-count sweep (Track A).** Table: steps (5/15/25) x the four objective metrics. At what step count did quality visibly plateau? Was there a step count where the clip was clearly still under-denoised (describe what that looked like)?

**4c. Guidance-scale sweep (Track A).** Table: guidance (1.0/9.0/15.0) x the four objective metrics + your prompt-adherence judgment (1-5). Did high guidance visibly over-sharpen or flicker, as the intro predicted? Cite the frame(s) where you saw it, if any.

**4d. Latent-space interpolation (Track A).** Embed the t=0/0.5/1.0 comparison image. Is t=0.5 a plausible "in-between" scene, or does it look unrelated to both endpoints? What does your answer imply about whether this model's noise space is smoothly organized? **If the cell errored for you:** paste the exact error and stop there — do not describe an expected result you did not observe.

**Track B (no GPU):** do 4a only (it runs on CPU). For 4b-4d, write 2-3 sentences on what you would predict WOULD happen for each control based on the lecture's diffusion-mechanics material, explicitly labeled as an untested prediction, not an observation.

## AI-Agent Usage Disclosure

State:

- which tools you used
- what they helped produce
- what you verified or rewrote yourself
- one specific thing you did not trust without checking